<center>
<a href="http://www.insa-toulouse.fr/" ><img src="http://www.math.univ-toulouse.fr/~besse/Wikistat/Images/logo-insa.jpg" style="float:left; max-width: 120px; display: inline" alt="INSA"/></a> 

</center>

# Machine Learning Project
MoraGarcia Carmen, De Artiagoitia Léo, Dubouchet Chloé et Perrin Alicia


## Introduction

The data is taken from the KAGGLE competition website; it is the data set "Cardivascular Disease Risk
Prediction Dataset" available here: https://www.kaggle.com/datasets/bertnardomariouskono/cardiovasculardisease-risk-prediction-dataset.


This dataset contains 15,000 synthetic patient medical records specifically designed to predict the risk of
cardiovascular disease. Although synthetic, the data is generated using medical heuristics to ensure realistic
correlations between variables, such as the relationship between age, BMI, and blood pressure. The dataset
includes 19 variables for 15,000 patients.

**WARNING** In order to have exactly the same training and test samples in both languages ​​so that their results can be compared afterwards, it is imperative to run the python notebook before this one.

In [2]:
# Chargement des librairies nécessaires
options(warn = -1)
library(ggplot2)
library(tidyverse)
library(gridExtra)
library(GGally)
library(plotly)
library(corrplot)
library(reshape2)
library(FactoMineR) 
library(factoextra)
library(glmnet) 
library(ggfortify)
library(pROC)
library(ROCR)

## 1. Exploratory data analysis

## a. Read the table of data

In [13]:
df<-read.csv("archive/healthcare_synthetic_data.csv", sep=",")
head(df)

,Patient_ID,Age,Gender,Height_cm,Weight_kg,BMI,Systolic_BP,Diastolic_BP,Cholesterol_Total,Cholesterol_LDL,Cholesterol_HDL,Fasting_Blood_Sugar,Smoking_Status,Alcohol_Consumption,Physical_Activity_Level,Family_History,Stress_Level,Sleep_Hours,Heart_Disease_Risk
,<chr>,<int>,<int>,<dbl>,<dbl>,<dbl>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>
1,PID-00001,60,0,146.9,51.3,23.8,140,89,217,151,52,83,0,1,3,0,1,8,0
2,PID-00002,53,0,161.8,76.6,29.3,128,81,203,119,38,116,0,0,1,0,7,9,0
3,PID-00003,62,1,174.7,92.4,30.3,141,100,173,124,45,90,0,0,0,0,1,7,1
4,PID-00004,73,1,173.3,68.9,22.9,136,96,193,117,45,81,0,0,1,0,2,7,1
5,PID-00005,52,1,178.6,79.8,25.0,122,80,236,153,41,79,0,1,2,0,2,6,0
6,PID-00006,52,0,159.6,60.3,23.7,134,92,225,155,48,103,0,0,1,1,4,8,0


We start by checking the nature of the different variables and their encoding.

In [14]:
df$Gender <- factor(df$Gender, labels = c("Woman", "Man"))

df$Smoking_Status <- factor(df$Smoking_Status, labels = c("Non_Smoker", "Smoker"))

df$Alcohol_Consumption <- factor(df$Alcohol_Consumption, labels = c("Non", "Moderate", "Heavy"))

df$Physical_Activity_Level <- factor(df$Physical_Activity_Level)

df$Family_History <- factor(df$Family_History, labels = c("No", "Yes"))

df$Heart_Disease_Risk <- factor(df$Heart_Disease_Risk, labels = c("Low", "High"))

Secondly, we transform some variable to be Gaussian.

In [15]:
df <- df %>%
  mutate(
    R_Weight_kg = sqrt(Weight_kg),
    R_BMI = sqrt(BMI)
  )

variables <- c('Age', 'Height_cm', 'R_Weight_kg', 'R_BMI', 'Systolic_BP', 
               'Diastolic_BP', 'Cholesterol_Total', 'Cholesterol_LDL', 
               'Cholesterol_HDL', 'Fasting_Blood_Sugar', 'Stress_Level', 
               'Sleep_Hours')

df <- df %>% 
  select(-BMI, -Weight_kg)

summary(df)

  Patient_ID             Age          Gender       Height_cm    
 Length:15000       Min.   :25.00   Woman:7622   Min.   :138.5  
 Class :character   1st Qu.:46.00   Man  :7378   1st Qu.:158.5  
 Mode  :character   Median :55.00                Median :164.7  
                    Mean   :54.54                Mean   :165.3  
                    3rd Qu.:63.00                3rd Qu.:172.0  
                    Max.   :85.00                Max.   :198.1  
  Systolic_BP     Diastolic_BP    Cholesterol_Total Cholesterol_LDL
 Min.   : 90.0   Min.   : 60.00   Min.   :127.0     Min.   : 70.0  
 1st Qu.:127.0   1st Qu.: 85.00   1st Qu.:201.0     1st Qu.:128.0  
 Median :135.0   Median : 91.00   Median :216.0     Median :140.0  
 Mean   :135.1   Mean   : 90.54   Mean   :216.2     Mean   :140.4  
 3rd Qu.:143.0   3rd Qu.: 96.00   3rd Qu.:231.0     3rd Qu.:152.0  
 Max.   :182.0   Max.   :120.00   Max.   :303.0     Max.   :210.0  
 Cholesterol_HDL Fasting_Blood_Sugar    Smoking_Status  Alcohol_Consu

# 2. Prediction of Heart’s Disease Risk
We consider the problem of predicting the variable Heart_Disease_Risk from the other variables from a machine learning point of view, i.e. focusing on model performance. The aim is to determine the best performance we can expect, and which models achieve it.

## a. Importation of training sample and a test sample

We decided to export the indices of the individuals in each sample so that we could compare the results obtained in R and in Python on the same data.

In [18]:
train_idx <- read.csv("train_indices.csv")$index
test_idx <- read.csv("test_indices.csv")$index

train_idx_r <- train_idx + 1
test_idx_r <- test_idx + 1

df_train <- df[train_idx_r, ]
df_test <- df[test_idx_r, ]

In [ ]:
X_train <- df_train[, !(names(df_train) %in% c("Patient_ID", "Heart_Disease_Risk"))]
Y_train <- df_train$Heart_Disease_Risk

X_test <- df_test[, !(names(df_test) %in% c("Patient_ID", "Heart_Disease_Risk"))]
Y_test <- df_test$Heart_Disease_Risk

quanti <- c("Age", "Height_cm", "R_Weight_kg", "R_BMI", "Systolic_BP", "Diastolic_BP", "Cholesterol_Total", "Cholesterol_LDL", "Cholesterol_HDL", "Fasting_Blood_Sugar", "Stress_Level", "Sleep_Hours")

X_train_scaled <- scale(X_train[, quanti])
X_test[, quanti] <- scale(X_test[, quanti], 
                          center = attr(X_train_scaled, "scaled:center"), 
                          scale  = attr(X_train_scaled, "scaled:scale"))
X_train[, quanti] <- X_train_scaled

Xr_train <- scale(X_train[, quanti])
Xr_test <- scale(X_test[, quanti])

Xr_train <- as.data.frame(Xr_train)
Xr_test <- as.data.frame(Xr_test)

## b. Comparison of forecasting models

### i) Linear model

### ii)  SVR/SVM

### iii) Optimal tree

### iv) Model Aggregation : Random forest

We will create a model for predicting the risk of heart disease using a random forest.

In [ ]:
library(randomForest)

In [ ]:
rf.heart <- randomForest(
  x = X_train,
  y = Y_train,
  xtest = X_test,
  ytest = Y_test,
  ntree = 500,
  do.trace = 50,
  importance = TRUE,
)

attributes(rf.heart)


To have better results, we search for the best value for mtry.

In [ ]:
best_mtry <- tuneRF(X_train, Y_train, stepFactor=1.5, improve=0.01, ntreeTry=500)
print(best_mtry)

The optimal value is 4. Therefore, we need to restart the learning process.

In [ ]:
rf.heart <- randomForest(
  x = X_train,
  y = Y_train,
  xtest = X_test,
  ytest = Y_test,
  ntree = 500,
  do.trace = 50,
  importance = TRUE,
  mtry=4
)

In [ ]:
sort(round(importance(rf.heart), 2)[,4], decreasing=TRUE)
varImpPlot(rf.heart)

We observe that the variable that contributes the most is Smoking Status, which corresponds to the results of the descriptive analysis.

In [ ]:
pred.heart=rf.heart$test$predicted


matrice <- table(pred.heart,Y_test)

print(matrice)

accuracy <- sum(diag(matrice)) / sum(matrice)
print(paste("Accuracy :", round(accuracy * 100, 2), "%"))

At the end, we obtaint 72.9% of accuracy.

### v) Model Aggregation : Boosting

When building an aggregation model using boosting, you can adjust the `shrinkage` parameter, which corresponds to the learning rate. More precisely, boosting is a method where new prediction trees are sequentially added to correct the errors made by all the previous trees. The shrinkage parameter controls the contribution of each new tree to the final model. Instead of adding 100% of the new tree's prediction, only a fraction is added (for example, 10% or 1%), thus limiting overfitting.
We can also adjust the `ntree` parameter, which corresponds to the final number of trees.
We will attempt to optimize these parameters through cross-validation.

In [ ]:
library(gbm)

In [ ]:
target_boost <- as.numeric(Y_train) - 1
df_boost_train <- cbind(X_train, Heart_Disease_Risk = target_boost)

boost.dis <- gbm(Heart_Disease_Risk ~ ., 
                 data = df_boost_train, 
                 distribution = "adaboost", 
                 n.trees = 2000, 
                 cv.folds = 10,
                 n.minobsinnode = 5,
                 shrinkage = 0.03,
                 verbose = FALSE)


plot(boost.dis$cv.error, type = "l", 
     main = "CV error as a function of the number of trees",
     xlab = "Number of iterations", ylab = "Error")

In [ ]:
print(boost.dis$cv.error[1])

In [ ]:
# optimal number of iterations 
best.ited=gbm.perf(boost.dis,method="cv")

This graph shows that the optimal number of iterations is around 1000. However, this result assumes a fixed shrinkage. Therefore, we will look for the best parameter combination in the next cell.

In [ ]:
shrink_vals <- c(0.001, 0.01, 0.05, 0.1)
results <- data.frame()

for (s in shrink_vals) {
  m <- gbm(Heart_Disease_Risk ~ ., data = df_boost_train, distribution = "adaboost",
           n.trees = 1000, shrinkage = s, cv.folds = 10)
  
  best_t <- gbm.perf(m, method = "cv", plot.it = FALSE)
  results <- rbind(results, data.frame(shrinkage = s, n.trees = best_t, error = m$cv.error[best_t]))
}

best_combi=results[order(results$error), ][1, ]

In [ ]:
print(best_combi)

Optimizing these two parameters yields a better result: 0.83 < 0.99.

In [ ]:
prob_test <- predict(boost.dis, newdata = X_test, n.trees = 307, shrinkage=0.1, type = "response")

pred_test <- ifelse(prob_test > 0.5, 1, 0)

matrice <- table(Reel = Y_test, Predi = pred_test)

print(matrice)

accuracy <- sum(diag(matrice)) / sum(matrice)
print(paste("Accuracy :", round(accuracy * 100, 2), "%"))

Finally, we obtaint 72.57% of accuracy.

### vi) Neural network

### vii) ROC curve

# 3. Prédiction of Cholesterol_LDL

Repeat the previous steps to predict the variable Cholesterol_LDL from all the other variables (except the
variable Heart_Disease_Risk).

## a. Creation of training sample and a test sample

In [ ]:
XC_train <- df_train[, !(names(df_train) %in% c("Patient_ID", "Heart_Disease_Risk", "Cholesterol_LDL"))]
YC_train <- df_train$Cholesterol_LDL

XC_test <- df_test[, !(names(df_test) %in% c("Patient_ID", "Heart_Disease_Risk","Cholesterol_LDL"))]
YC_test <- df_test$Cholesterol_LDL

quanti <- c("Age", "Height_cm", "R_Weight_kg", "R_BMI", "Systolic_BP", "Diastolic_BP", "Cholesterol_Total", "Cholesterol_HDL", "Fasting_Blood_Sugar", "Stress_Level", "Sleep_Hours")

XC_train_scaled <- scale(XC_train[, quanti])
XC_test[, quanti] <- scale(XC_test[, quanti], 
                          center = attr(XC_train_scaled, "scaled:center"), 
                          scale  = attr(XC_train_scaled, "scaled:scale"))
XC_train[, quanti] <- XC_train_scaled

XCr_train <- scale(XC_train[, quanti])
XCr_test <- scale(XC_test[, quanti])

XCr_train <- as.data.frame(XCr_train)
XCr_test <- as.data.frame(XCr_test)

## b. Comparison of forecasting models

### i) Linear model

Before we begin, we retrieved the gplot.res function, which allows us to plot the residual graph with fixed colors and scales on the axes. This will highlight the effectiveness of the models that will follow.

In [ ]:
gplot.res <- function(x, y, xmin, xmax, ymin, ymax, titre = "titre"){
    ggplot(data.frame(x=x, y=y),aes(x,y))+
    geom_point(col = "blue")+xlim(xmin, xmax)+ylim(ymin, ymax)+
    ylab("Residues")+ xlab("Predicted values")+
    ggtitle(titre)+
    geom_hline(yintercept = 0,col="green")
}

### ii)  SVR/SVM

### iii) Optimal tree

### iv) Model Aggregation : Random forest

We will now create a prediction of LDL cholesterol levels using a random forest.

In [ ]:
rf_chol <- randomForest(
  x = XC_train,
  y = YC_train,
  ntree = 500,
  importance = TRUE,
  keep.forest = TRUE
)

In [ ]:
y_pred <- predict(rf_chol, XC_test)

sse <- sum((YC_test - y_pred)^2)
sst <- sum((YC_test - mean(YC_test))^2)
r2_test <- 1 - (sse / sst)
cat(sprintf("R² coefficient on the test set: %.4f\n", r2_test))


print("Importance of variables :")
print(importance(rf_chol))
cat("Number of input variables :", ncol(XC_train), "\n")
oob_err <- 1 - tail(rf_chol$rsq, 1)
cat(sprintf("Error OOB : %.4f\n", oob_err))
cat(sprintf("Error OOB of the test sample : %.4f\n", 1 - r2_test))

Finding the best value for max_features using cross-validation:

- If max_features is too low: The trees are very different but they are "blind" (they don't have enough information to make a good decision).

- If max_features is too high: The trees are very intelligent individually, but they all do the same thing.

In [ ]:
# /!\ take a long time to run
best_mtry <- tuneRF(XC_train, YC_train, stepFactor=1.5, improve=0.01, ntreeTry=500)
print(best_mtry)

We find mtry optimal at 7 so we start again with the correct parameter.

In [ ]:
# /!\ take a long time to run
rf_chol_Opt <- randomForest(
  x = XC_train,
  y = YC_train,
  ntree = 500,
  importance = TRUE,
  keep.forest = TRUE,
  mtry=7
)

In [ ]:
fit_rf_chol <- rf_chol_Opt$predicted
res.rfr <- YC_train - fit_rf_chol
gplot.res(fit_rf_chol, res.rfr, 80, 200, -45, 45, titre = "Residuals - Random Forest")

The cloud is homogeneous, thick in the middle, and thins out symmetrically. It is pure white noise. This proves that the Random Forest has captured almost all the logical structure of the data, leaving only randomness behind.

In [ ]:
y_pred_Opt <- predict(rf_chol_Opt, XC_test)

sse <- sum((YC_test - y_pred_Opt)^2)
sst <- sum((YC_test - mean(y_pred_Opt))^2)
r2_test <- 1 - (sse / sst)
cat(sprintf("R² coefficient on the test set : %.4f\n", r2_test))

print("Importance of variables :")
print(importance(rf_chol_Opt))

In [ ]:
cat("Number of input variables :", ncol(XC_train), "\n")
oob_err <- 1 - tail(rf_chol_Opt$rsq, 1)
cat(sprintf("Optimal OOB error : %.4f\n", oob_err))
cat(sprintf("Optimal OOB error of the test sample : %.4f\n", 1 - r2_test))

With the optimal model, we finally obtain an OOB error on the test sample of 0.32.

### v) Model Aggregation : Boosting

In this section we will create a prediction model using boosting.

In [ ]:
XC_boost_train <- df_train[, !(names(df_train) %in% c("Patient_ID", "Heart_Disease_Risk"))]
YC_boost_train <- df_train$Cholesterol_LDL

XC_boost_test <- df_test[, !(names(df_test) %in% c("Patient_ID", "Heart_Disease_Risk"))]
YC_boost_test <- df_test$Cholesterol_LDL

quanti <- c("Age", "Height_cm", "R_Weight_kg", "R_BMI", "Systolic_BP", "Diastolic_BP", "Cholesterol_Total", "Cholesterol_HDL", "Fasting_Blood_Sugar", "Stress_Level", "Sleep_Hours")

XC_boost_train_scaled <- scale(XC_boost_train[, quanti])
XC_boost_test[, quanti] <- scale(XC_boost_test[, quanti], 
                          center = attr(XC_boost_train_scaled, "scaled:center"), 
                          scale  = attr(XC_boost_train_scaled, "scaled:scale"))
XC_boost_train[, quanti] <- XC_train_scaled

In [ ]:
boost.reg = gbm(Cholesterol_LDL ~ ., data = XC_boost_train, distribution = "gaussian", n.trees = 400, 
    cv.folds = 5, n.minobsinnode = 5, shrinkage = 0.1, verbose = FALSE)

plot(boost.dis$cv.error, type = "l", 
     main = "CV error as a function of the number of trees",
     xlab = "Number of iterations", ylab = "Error")

In [ ]:
best.iter=gbm.perf(boost.reg,method="cv")

As in classification, we will choose the best combination of parameters. The preceding cells only help us to get an idea of ​​the orders of the scale.

In [ ]:
shrink_vals <- c(0.001, 0.01, 0.05, 0.1)
results <- data.frame()

for (s in shrink_vals) {
  m <- gbm(Cholesterol_LDL ~ ., data = X_train, distribution = "gaussian",
           n.trees = 600, shrinkage = s, cv.folds = 10)
  
  best_t <- gbm.perf(m, method = "cv", plot.it = FALSE)
  results <- rbind(results, data.frame(shrinkage = s, n.trees = best_t, error = m$cv.error[best_t]))
}

best_combi=results[order(results$error), ][1, ]

In [ ]:
print(best_combi)

In [ ]:
fit.boostr=boost.reg$fit
res.boostr=fit.boostr-XC_boost_train[,"Cholesterol_LDL"]
gplot.res(fit.boostr,res.boostr, 80, 200, -45, 45, titre="")

The cloud shape is good. The model works well.

In [ ]:
pred_boost <- predict(boost.reg, newdata = XC_boost_test, n.trees = 283, shrinkage=0.05, type = "response")
err=sum((pred_boost-XC_boost_test[,"Cholesterol_LDL"])^2)/nrow(XC_boost_test)
var_test = var(XC_boost_test[,"Cholesterol_LDL"])

print(paste("The squared error on the test sample of this model is", err, "."))
print(paste("Total variance :", var_test))
print(paste("Error margin :", err / var_test))

Finally, we obtain an error of 0.31.

### vi) Neural network

### vii) ROC curve